[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# *LangGraph*: estado, grafos y ejecución durable.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 6: *LangGraph: estado, durabilidad y multiagente*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se construirán grafos con *LangGraph*: estado y reductores, nodos, aristas condicionales, *checkpointers*, interrupciones, reanudación tras una falla y viaje en el tiempo. Se termina con el flujo del asistente de soporte como grafo explícito.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a un modelo de lenguaje: las respuestas varían entre ejecuciones y consumen tokens de tu clave.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (ejecución local), definirá el modelo del curso y verificará la clave de API.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if not os.environ.get("GOOGLE_API_KEY") and MODELO.startswith("google_genai"):
    raise RuntimeError("Falta GOOGLE_API_KEY: agrégala como secreto de Codespaces y reconstruye el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

import aiosqlite
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel

---
## 1. Estado, nodos y aristas.

Un grafo de *LangGraph* tiene tres elementos:
* **Estado:** un diccionario tipado que comparten todos los nodos.
* **Nodos:** funciones que reciben el estado y devuelven **solo las claves que cambian**.
* **Aristas:** qué nodo sigue; las **condicionales** deciden con una función.

El primer grafo no usa modelos: así se ve la mecánica sin ruido.

* La siguiente celda construirá un grafo de cuatro nodos sin modelos y lo ejecutará.

In [ ]:
class EstadoDemo(TypedDict):
    texto: str
    pasos: Annotated[list[str], operator.add]   # reductor: las listas se concatenan


def limpiar(estado):
    return {"texto": estado["texto"].strip().lower(), "pasos": ["limpiar"]}


def contar(estado):
    return {"pasos": [f"contar: {len(estado['texto'].split())} palabras"]}


def ruta(estado) -> str:
    return "largo" if len(estado["texto"]) > 30 else "corto"


def largo(estado):
    return {"pasos": ["ruta larga"]}


def corto(estado):
    return {"pasos": ["ruta corta"]}


g = StateGraph(EstadoDemo)
for nodo in (limpiar, contar, largo, corto):
    g.add_node(nodo.__name__, nodo)
g.add_edge(START, "limpiar")
g.add_edge("limpiar", "contar")
g.add_conditional_edges("contar", ruta, {"largo": "largo", "corto": "corto"})
g.add_edge("largo", END)
g.add_edge("corto", END)
demo = g.compile()
demo.invoke({"texto": "  Hola Mundo  ", "pasos": []})

**Observa:** `pasos` acumuló lo que devolvió cada nodo gracias al **reductor** `operator.add`; `texto` se sobrescribió porque no tiene reductor. `MessagesState` usa un reductor especial que agrega mensajes.

---
## 2. *Checkpointers*, interrupciones y viaje en el tiempo.

* La siguiente celda agregará un nodo que se **interrumpe** para pedir una decisión humana.

In [ ]:
def revisar(estado):
    decision = interrupt({"pregunta": f"¿Aprobar el texto '{estado['texto']}'?"})
    return {"pasos": [f"revisión: {decision}"]}


g2 = StateGraph(EstadoDemo)
g2.add_node("limpiar", limpiar)
g2.add_node("revisar", revisar)
g2.add_edge(START, "limpiar")
g2.add_edge("limpiar", "revisar")
g2.add_edge("revisar", END)
demo2 = g2.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "demo-1"}}
r = demo2.invoke({"texto": " Propuesta ", "pasos": []}, cfg)
r["__interrupt__"]

* La siguiente celda reanudará el hilo con la decisión.

In [ ]:
demo2.invoke(Command(resume="aprobado"), cfg)

* La siguiente celda listará todos los *checkpoints* del hilo, del más reciente al más antiguo.

In [ ]:
[(p.next, p.values.get("pasos")) for p in demo2.get_state_history(cfg)]

**💡 Ideas clave de la sección 2:**
* El *checkpointer* guarda el estado después de cada paso, por hilo (`thread_id`).
* `interrupt()` pausa el grafo; `Command(resume=...)` lo reanuda con un valor.
* El historial de *checkpoints* permite inspeccionar y volver a ejecutar desde cualquier paso.

---
## 3. El asistente de soporte como grafo.

```
inicio → clasificar ─┬─ duda_politica ─→ politicas → fin
                     ├─ incidente normal ─→ tickets → fin
                     └─ incidente urgente → aprobar ─┬─ sí → tickets → fin
                                                     └─ no → rechazado → fin
```
Los nodos `politicas` y `tickets` son agentes de `create_agent` usados como **subagentes**.

* La siguiente celda definirá las herramientas de políticas.

In [ ]:
from langchain.tools import tool


@tool
def listar_politicas() -> list[str]:
    """Lista los nombres de las políticas internas disponibles."""
    return sorted(f[:-3] for f in os.listdir(POLITICAS) if f.endswith(".md"))


@tool
def leer_politica(nombre: str) -> str:
    """Devuelve el texto completo de una política interna.

    Args:
        nombre: nombre exacto devuelto por listar_politicas, sin extensión.
    """
    ruta = os.path.join(POLITICAS, f"{nombre}.md")
    if not os.path.isfile(ruta):
        return f"No existe la política '{nombre}'. Disponibles: {', '.join(listar_politicas.invoke({}))}."
    with open(ruta, encoding="utf-8") as archivo:
        return archivo.read()

* La siguiente celda conectará el servidor MCP de tickets.

In [ ]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

cliente_tickets = MultiServerMCPClient({
    "tickets": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [os.path.join(RAIZ, "mcp_servers", "tickets_server.py")],
    }
})
herramientas_mcp = await cliente_tickets.get_tools()
[h.name for h in herramientas_mcp]

* La siguiente celda definirá el estado, el clasificador, los subagentes y los nodos.

In [ ]:
class Estado(MessagesState):
    tipo: Literal["duda_politica", "incidente", "otro"] | None
    prioridad: Literal["baja", "normal", "urgente"] | None
    aprobado: bool | None


class Clasificacion(BaseModel):
    tipo: Literal["duda_politica", "incidente", "otro"]
    prioridad: Literal["baja", "normal", "urgente"]


clasificador = init_chat_model(MODELO).with_structured_output(Clasificacion)
agente_politicas = create_agent(MODELO, tools=[listar_politicas, leer_politica], name="politicas",
                                system_prompt="Responde dudas solo con base en las políticas internas.")
agente_tickets = create_agent(MODELO, tools=herramientas_mcp, name="tickets",
                              system_prompt="Gestiona tickets. El usuario actual es 'ana.lopez'.")
FALLAR = {"activo": False}   # interruptor para la sección 4


async def clasificar(estado: Estado):
    c = await clasificador.ainvoke(estado["messages"])
    return {"tipo": c.tipo, "prioridad": c.prioridad}


def aprobar(estado: Estado):
    decision = interrupt({"pregunta": "¿Aprobar el ticket urgente?", "mensaje": estado["messages"][-1].text})
    return {"aprobado": decision == "si"}


async def tickets(estado: Estado):
    if FALLAR["activo"]:
        raise RuntimeError("Falla simulada del sistema de tickets")
    r = await agente_tickets.ainvoke({"messages": estado["messages"]})
    return {"messages": r["messages"][-1:]}


async def politicas(estado: Estado):
    r = await agente_politicas.ainvoke({"messages": estado["messages"]})
    return {"messages": r["messages"][-1:]}


def rechazado(estado: Estado):
    return {"messages": [{"role": "assistant", "content": "Tu solicitud quedó en revisión con un agente humano."}]}


def ruta_inicial(estado: Estado) -> str:
    if estado["tipo"] == "incidente":
        return "aprobar" if estado["prioridad"] == "urgente" else "tickets"
    return "politicas"

* La siguiente celda armará el grafo con un *checkpointer* en *SQLite* (persiste en `data/checkpoints.db`).

In [ ]:
grafo = StateGraph(Estado)
for nombre, nodo in [("clasificar", clasificar), ("aprobar", aprobar), ("tickets", tickets),
                     ("politicas", politicas), ("rechazado", rechazado)]:
    grafo.add_node(nombre, nodo)
grafo.add_edge(START, "clasificar")
grafo.add_conditional_edges("clasificar", ruta_inicial, ["aprobar", "tickets", "politicas"])
grafo.add_conditional_edges("aprobar", lambda e: "tickets" if e["aprobado"] else "rechazado", ["tickets", "rechazado"])
for nodo in ["tickets", "politicas", "rechazado"]:
    grafo.add_edge(nodo, END)

conexion = await aiosqlite.connect(os.path.join(DATA, "checkpoints.db"))
app = grafo.compile(checkpointer=AsyncSqliteSaver(conexion))
print(app.get_graph().draw_mermaid())

* La siguiente celda ejecutará un caso urgente con *streaming* de pasos; debe detenerse en `aprobar`.

In [ ]:
config = {"configurable": {"thread_id": "nb09-urgente"}}
entrada = {"messages": [{"role": "user", "content": "Perdí mi laptop con datos de clientes, ¡urgente!"}]}
async for evento in app.astream(entrada, config, stream_mode="updates"):
    print(evento)

* La siguiente celda reanudará con la aprobación.

In [ ]:
async for evento in app.astream(Command(resume="si"), config, stream_mode="updates"):
    print(evento)

---
## 4. Falla y reanudación.

Se activa una falla en el nodo `tickets`. Como cada paso queda guardado, al reanudar **el mismo hilo** con `None` como entrada, el grafo continúa desde el último *checkpoint* sin repetir la clasificación.

* La siguiente celda provocará la falla y mostrará el nodo pendiente.

In [ ]:
FALLAR["activo"] = True
config_f = {"configurable": {"thread_id": "nb09-falla"}}
try:
    await app.ainvoke({"messages": [{"role": "user", "content": "El monitor parpadea a veces, ¿me abres un ticket normal?"}]}, config_f)
except RuntimeError as e:
    print("Falló:", e)
(await app.aget_state(config_f)).next

* La siguiente celda quitará la falla y reanudará el hilo.

In [ ]:
FALLAR["activo"] = False
(await app.ainvoke(None, config_f))["messages"][-1].text

**💡 Ideas clave de la sección 4:**
* La ejecución durable permite reanudar tras una falla sin repetir pasos ya hechos.
* Un grafo explícito hace auditable el flujo: qué ruta tomó cada caso y por qué.

### Ejercicio 1 · Ruta `otro`.

Agrega un nodo `fuera_de_alcance` que responda `"Solo puedo ayudarte con soporte interno."` **sin llamar al modelo**, y haz que `ruta_inicial` envíe ahí los casos `tipo == "otro"`. Compila como `app_2` (sin *checkpointer*) y prueba con `"¿Quién ganó el partido ayer?"`.

In [ ]:
# TU CÓDIGO AQUÍ

### Ejercicio 2 · *Streaming* de tokens.

Ejecuta una duda de política con `stream_mode="messages"` y muestra solo el texto de cada fragmento a medida que llega (pista: cada evento es una tupla `(fragmento, metadatos)`).

In [ ]:
# TU CÓDIGO AQUÍ

### Reto 1 · Volver a un *checkpoint*.

En el hilo `nb09-urgente`, encuentra el *checkpoint* en el que el grafo estaba detenido en `aprobar` y reanuda **desde ahí** con `"no"` (pista: pasa el `config` de ese *checkpoint* a `app.ainvoke(Command(resume="no"), config_del_checkpoint)`). ¿Qué rama se ejecuta?

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## ✅ Cierre.

**Práctica:** terminar los ejercicios 1 y 2.

**Siguiente *notebook*:** `10_patrones_multiagente.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>